# LE9 - Diffusion in Football Networks

**Diffusion** describes the process of information or knowledge spreading through interactions in a network, from an initial sender (or small group of senders) to final receivers.

### Applications in Football Networks

**Pass Networks (Players)**:
- Model how tactical information spreads through a team
- Identify which players are key for spreading tactics/playing style
- Understand how formations and strategies propagate during a match

**Match Networks (Teams)**:
- Analyze how playing styles or tactical innovations spread between teams
- Track how competitive strategies diffuse through tournaments
- Identify influential teams that set trends

### Rogers' Innovation Diffusion Theory

Everett Rogers' theory (1962) describes how innovations spread through populations in five adopter categories:
1. **Innovators (2.5%)**: First adopters, risk-takers, well-connected
2. **Early Adopters (13.5%)**: Opinion leaders, high social status, selective
3. **Early Majority (34%)**: Deliberate adopters, contact with early adopters
4. **Late Majority (34%)**: Skeptical, adopt after average person
5. **Laggards (16%)**: Traditional, resistant to change

This creates an S-curve of adoption over time.

### Our Approach

We will implement and analyze diffusion models to understand:
1. How behaviors/tactics spread through player networks
2. Which players/teams are most influential in spreading information
3. How to maximize reach with limited initial activations
4. Different diffusion dynamics in football networks

## 2. Independent Cascade Model (ICM)

The **Independent Cascade Model** models information transmission through direct connections to neighbor nodes.

### Core Assumptions

1. A node can only influence direct neighbors (respecting edge direction in directed graphs)
2. Nodes can only be **active** or **inactive** 
3. An activated node has **exactly one chance** to activate its neighbors (not at every iteration)
4. Nodes can only transition from inactive → active, never the reverse

### Intuition

Think of the ICM as a "one-shot influence" model:
- **Football analogy**: A player learns a new tactical approach and tries once to convince their passing partners to adopt it
- Each connection has a probability of successful influence
- Once you've tried to influence someone, you can't try again
- The behavior spreads in waves through the network

### Algorithm Steps

**Initialization:**
- Network with edges having attribute $p_{i,j}$ (activation probability)
- Set $A$ of initially activated nodes (seed set)

**Iteration Process:**

1. For each node $i$ in current active set $A$:
   - For each inactive neighbor $j$:
     - Generate random number $r \in [0,1]$
     - If $r < p_{i,j}$, activate neighbor $j$
     - Add $j$ to new active set $A_{new}$

2. If $A_{new}$ is empty → **stop**
   
   Otherwise: $A = A_{new}$ and repeat step 1

3. Each node attempts activation of neighbors **exactly once**

### Mathematical Representation

For an edge from node $i$ to node $j$:

$$P(\text{activation}) = \begin{cases} 
1 & \text{if } r_{i,j} < p_{i,j} \\
0 & \text{otherwise}
\end{cases}$$

where $r_{i,j} \sim \text{Uniform}(0,1)$ is a random number generated once per edge.

### Activation Probabilities

How to define $p_{i,j}$? Several approaches:

1. **Random values**: Kempe et al.'s original suggestion
2. **Edge weights**: Use normalized pass frequency/match count
3. **Node similarity**: Use assortativity metrics (how similar nodes are)
4. **Historical data**: Learn from past diffusion events

**In our football networks:**
- **Pass network**: Use passing frequency as proxy for influence probability
- **Match network**: Use match count or competitive similarity

## 3. Maximizing Spread with Greedy Algorithm

### The Influence Maximization Problem

**Goal**: Given budget $k$ (number of initial activations), find which $k$ nodes to activate initially to maximize final spread.

**Notation:**
- $S$ = seed set (initially activated nodes)
- $f(S)$ = final number of activated nodes
- $k$ = budget (size of seed set)
- Goal: Find $S$ with $|S| = k$ such that $f(S)$ is maximized

### Challenge

This problem is **NP-hard** (computationally intractable to solve optimally). We use a **Greedy Algorithm** instead.

### Greedy Approach

**Greedy principle**: At each step, choose the option that provides the largest immediate gain.

**Key modification for ICM**: To make the algorithm deterministic, we pre-compute random numbers $r_{i,j}$ for all edges once during initialization, rather than regenerating them each time.

### Algorithm Steps

**Initialization:**
- Network with activation probabilities $p_{i,j}$ and pre-computed random values $r_{i,j}$
- Empty seed set $S = \{\}$
- Budget $k$
- Counter $i = 0$

**Greedy Selection Loop** (while $i < k$):

1. For each node $v \notin S$:
   - Calculate $f(S \cup \{v\})$ (spread if $v$ added to seed set)
   
2. Select node $v^*$ with maximum $f(S \cup \{v^*\})$
   
3. Add $v^*$ to $S$: $S = S \cup \{v^*\}$
   
4. Increment: $i = i + 1$

5. Return final seed set $S$

### Example Calculation

Given network with activation function: $|i - j| \equiv 2 \pmod{3}$

**First iteration** ($S = \{\}$, $k=2$):
- Calculate: $f(\{1\}) = 4$, $f(\{2\}) = 1$, $f(\{3\}) = 1$, $f(\{4\}) = 2$, $f(\{5\}) = 1$, $f(\{6\}) = 4$
- Maximum: nodes 1 or 6 both achieve $f = 4$
- Choose node 6: $S = \{6\}$

**Second iteration** ($S = \{6\}$):
- Calculate: $f(\{6,3\}) = 5$, $f(\{6,5\}) = 5$
- Choose node 3: $S = \{6, 3\}$
- $|S| = k = 2$ → **stop**

Final result: $S = \{6, 3\}$ with spread $f(S) = 5$

### Interpretation for Football

**Pass networks**: Which players should receive tactical training first to maximize team-wide adoption?

**Match networks**: Which teams should be targeted for a new playing style to maximize diffusion through competitive play?

### Performance Note

Greedy algorithms are:
- Fast and practical
- Not guaranteed to find optimal solution
- Often provide good approximations (within 63% of optimal for ICM)

## 4. Voter Model

The **Voter Model** is a simple consensus model where opinions spread through random imitation.

### Core Mechanism

1. Each node has initial opinion: $+1$ (adopted) or $-1$ (not adopted)
2. **Iterative process**:
   - Select random node $v$
   - Node $v$ adopts opinion of random neighbor
3. Repeat until consensus (all nodes have same opinion)

### Mathematical Properties

**Consensus probability**: The probability that all nodes end with opinion $+1$ equals the initial fraction of nodes with opinion $+1$.

$$P(\text{final consensus } = +1) = \frac{|\{v : v_{initial} = +1\}|}{|V|}$$

### Key Characteristics

- **Memoryless**: No history, only current neighbor opinions matter
- **Random**: No preferential influence based on node properties
- **Guaranteed consensus**: Always reaches uniform state (in finite connected networks)
- **Initial conditions matter**: Final consensus reflects initial distribution

### Intuition in Football

**Pass networks**: 
- Players randomly adopt tactics from passing partners
- Eventually team converges to uniform tactical approach
- Initial adopters determine likely final state

**Limitations**: Too simplistic - doesn't account for:
- Player influence/authority differences
- Multiple simultaneous interactions
- Resistance to change

## 5. Threshold Model

The **Threshold Model** implements peer pressure: nodes adopt when enough neighbors have adopted.

### Core Mechanism

**Initial state:**
- Small set of nodes already activated (initial adopters)
- Each node $v$ has personal threshold $\theta_v \in [0,1]$

**Iteration process:**

For each inactive node $v$:
1. Calculate fraction of active neighbors: 
   $$f_v = \frac{|\{u \in N(v) : u \text{ is active}\}|}{|N(v)|}$$

2. If $f_v \geq \theta_v$: activate node $v$

3. Repeat until no new activations or max iterations reached

### Mathematical Definition

Node $v$ activates at time $t+1$ if:

$$\frac{\sum_{u \in N(v)} \mathbb{1}[\text{u active at time t}]}{|N(v)|} \geq \theta_v$$

where $\mathbb{1}[\cdot]$ is the indicator function.

### Threshold Interpretation

- **Low threshold** ($\theta_v \approx 0$): Easy to influence, early adopter
- **Medium threshold** ($\theta_v \approx 0.5$): Needs majority of neighbors
- **High threshold** ($\theta_v \approx 1$): Resistant, needs nearly all neighbors

### Key Differences from ICM

| Aspect | ICM | Threshold Model |
|--------|-----|-----------------|
| Influence attempts | One-time only | Every iteration |
| Activation criterion | Probabilistic ($r < p$) | Deterministic ($f \geq \theta$) |
| Neighbor influence | Independent per neighbor | Collective (fraction) |
| Guaranteed spread | No | Depends on thresholds |

### Intuition in Football

**Pass networks:**
- Player adopts new tactic when enough teammates use it
- Models **social pressure** and **conformity**
- Different players have different resistance levels

**Example thresholds:**
- $\theta = 0.3$: Innovative player (30% neighbors needed)
- $\theta = 0.7$: Conservative player (70% neighbors needed)


## 6. Comparing Diffusion Models

### Model Characteristics Summary

| Model | Type | Activation | Influence | Best For |
|-------|------|------------|-----------|----------|
| **ICM** | Probabilistic | One-shot | Independent | Viral spread, marketing |
| **Voter** | Random | Iterative | Imitative | Consensus formation |
| **Threshold** | Deterministic | Iterative | Collective | Peer pressure, social norms |

### Independent Cascade Model

**Strengths:**
- Models viral spreading realistically
- One-time influence matches many real scenarios
- Good for maximizing initial reach

**Limitations:**
- No repeated influence
- Requires probability estimation
- Random component makes results variable

**Football application:** Initial tactical innovation spread

### Voter Model

**Strengths:**
- Extremely simple
- Always reaches consensus
- Mathematically tractable

**Limitations:**
- Too simplistic for most real scenarios
- No node differentiation
- Ignores network structure influence

**Football application:** Simple opinion convergence

### Threshold Model

**Strengths:**
- Models peer pressure realistically
- Deterministic (reproducible)
- Captures resistance to adoption

**Limitations:**
- Requires threshold estimation
- May not reach full adoption
- Doesn't model probability/uncertainty

**Football application:** Team-wide tactical adoption requiring majority support

### Expected Behavior in Football Networks

**Pass Networks** (high clustering, tactical units):
- ICM: Rapid spread through tactical groups
- Threshold: Cluster-by-cluster adoption (defenders, midfield, forwards)
- Voter: Slow convergence to team consensus

**Match Networks** (lower density, competitive structure):
- ICM: Limited spread, depends on edge probabilities
- Threshold: May require very low thresholds for cascade
- Voter: Very slow consensus across all teams

### Influence Maximization

All models can use greedy algorithms to find influential seed nodes, but:
- **ICM**: Well-studied, proven approximation guarantees
- **Threshold**: More complex to compute expected spread
- **Voter**: Less relevant (random dynamics dominate)